# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, their `@id`s, and related columns.

In [ ]:
# List all record sets with their @id, name, and their fields' @ids
record_sets = metadata.record_sets
print("Available record sets and their fields:\n")
for rs in record_sets:
    print(f"- Record Set: @id={rs.id}, name={rs.name}, fields={[field.id for field in rs.fields]}")
    for field in rs.fields:
        columns = getattr(field, "columns", [])
        print(f"    - Field: @id={field.id}, name={field.name}, columns={[col.id for col in columns]}")

## 3. Data Extraction
Load data from all record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set using mlcroissant and pandas
dataframes = {}

# We'll automatically use all discovered record sets, referencing by @id
record_set_ids = [rs.id for rs in metadata.record_sets]
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    else:
        print("No records found for this record set.")

# For illustration, pick the first non-empty record set for EDA
main_record_set_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rs_id
        break
if main_record_set_id:
    print(f"Using record set {main_record_set_id} for further analysis.")
    print(f"Columns in DataFrame: {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No non-empty record set found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates removing outliers, transforming data distributions, and grouping data by key attributes to prepare it for further analysis.

Below, we attempt to select a numeric field to demonstrate these operations. **If needed, edit the variable assignments using the output from Sections 2 & 3, referencing fields by their `@id`.**

In [ ]:
# Ensure there's a data frame to operate on
import numpy as np

if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Try to find a likely numeric field
    numeric_field_id = None
    for col in df.columns:
        # Try to infer numeric columns
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id:
        print(f"Using numeric field: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype != object else 10
        # Remove NaN
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())
        # Normalization
        mean = filtered_df[numeric_field_id].mean()
        std = filtered_df[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Try grouping by another field if possible
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == object:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable text field for grouping found.")
    else:
        print("No numeric column found in the record set.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. The following example will plot a histogram of the selected numeric field if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and 'numeric_field_id' in locals() and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
This notebook demonstrated how to load, review, and perform basic exploratory analysis on the FAIR² Mass Spectrometry dataset using the `mlcroissant` library. Key findings will depend on your selections above. Continue with deeper domain-specific analysis or modeling as needed.